# Bài tập 5 — Mục 3: Khử lồng truy vấn và Gộp View
**Subquery Transformation & View Merging**

**Tài liệu:** R. Elmasri & S. B. Navathe, *Fundamentals of Database Systems*, 7th ed.

- Chương 18, mục 18.1.1: Semi-join / Anti-join — **trang PDF 690–691** (trang sách 659–660).
- Chương 19, mục 19.2.2: Nested Subquery Optimization — **trang PDF 733–735** (trang sách 702–704).
- Chương 19, mục 19.2.3: Subquery (View) Merging — **trang PDF 735–737** (trang sách 704–706).
- Chương 7, mục 7.3: View `WORKS_ON1` — **trang PDF 259–260** (trang sách 228–229).
- Bộ dữ liệu COMPANY: **Figure 5.5 (PDF 192), Figure 5.6 (PDF 193)**.

## 1. Khởi tạo COMPANY từ chính dữ liệu trong sách
Các bảng, tên thuộc tính và bản ghi sử dụng dưới đây lấy từ **Figure 5.6, trang sách 162 / trang PDF 193**. Chỉ chọn những cột cần thiết cho mục 3. Không tải dữ liệu bên ngoài và không tự tạo thêm nhân viên, phòng ban hay dự án.

In [ ]:
import sqlite3

conn = sqlite3.connect(":memory:")
conn.row_factory = sqlite3.Row
conn.execute("PRAGMA foreign_keys = ON")

def rows(sql, parameters=()):
    return [tuple(row) for row in conn.execute(sql, parameters).fetchall()]

def show(title, sql, parameters=()):
    cur = conn.execute(sql, parameters)
    values = [tuple(v) for v in cur.fetchall()]
    headers = [col[0] for col in cur.description]
    print("\n" + title)
    if not values:
        print("(0 dòng)")
        return values
    widths = [max(len(str(h)), *(len(str(v[i])) for v in values)) for i,h in enumerate(headers)]
    print(" | ".join(str(h).ljust(widths[i]) for i,h in enumerate(headers)))
    print("-+-".join("-"*w for w in widths))
    for value in values:
        print(" | ".join(str(v).ljust(widths[i]) for i,v in enumerate(value)))
    print(f"Tổng: {len(values)} dòng")
    return values

def show_plan(title, sql):
    print("\n" + title)
    for row in conn.execute("EXPLAIN QUERY PLAN " + sql):
        print("  ", row[3])

In [ ]:
conn.executescript("""
DROP VIEW IF EXISTS WORKS_ON1;
DROP TABLE IF EXISTS WORKS_ON;
DROP TABLE IF EXISTS PROJECT;
DROP TABLE IF EXISTS EMPLOYEE;
DROP TABLE IF EXISTS DEPARTMENT;

CREATE TABLE DEPARTMENT (
    Dname TEXT NOT NULL,
    Dnumber INTEGER PRIMARY KEY
);
CREATE TABLE EMPLOYEE (
    Fname TEXT NOT NULL,
    Lname TEXT NOT NULL,
    Ssn TEXT PRIMARY KEY,
    Salary INTEGER NOT NULL,
    Dno INTEGER NOT NULL,
    FOREIGN KEY (Dno) REFERENCES DEPARTMENT(Dnumber)
);
CREATE TABLE PROJECT (
    Pname TEXT NOT NULL,
    Pnumber INTEGER PRIMARY KEY
);
CREATE TABLE WORKS_ON (
    Essn TEXT NOT NULL,
    Pno INTEGER NOT NULL,
    Hours REAL,
    PRIMARY KEY (Essn, Pno),
    FOREIGN KEY (Essn) REFERENCES EMPLOYEE(Ssn),
    FOREIGN KEY (Pno) REFERENCES PROJECT(Pnumber)
);

INSERT INTO DEPARTMENT VALUES\n  ('Research', 5),\n  ('Administration', 4),\n  ('Headquarters', 1);
INSERT INTO EMPLOYEE VALUES\n  ('John', 'Smith', '123456789', 30000, 5),\n  ('Franklin', 'Wong', '333445555', 40000, 5),\n  ('Alicia', 'Zelaya', '999887777', 25000, 4),\n  ('Jennifer', 'Wallace', '987654321', 43000, 4),\n  ('Ramesh', 'Narayan', '666884444', 38000, 5),\n  ('Joyce', 'English', '453453453', 25000, 5),\n  ('Ahmad', 'Jabbar', '987987987', 25000, 4),\n  ('James', 'Borg', '888665555', 55000, 1);
INSERT INTO PROJECT VALUES\n  ('ProductX', 1),\n  ('ProductY', 2),\n  ('ProductZ', 3),\n  ('Computerization', 10),\n  ('Reorganization', 20),\n  ('Newbenefits', 30);
INSERT INTO WORKS_ON VALUES\n  ('123456789', 1, 32.5),\n  ('123456789', 2, 7.5),\n  ('666884444', 3, 40.0),\n  ('453453453', 1, 20.0),\n  ('453453453', 2, 20.0),\n  ('333445555', 2, 10.0),\n  ('333445555', 3, 10.0),\n  ('333445555', 10, 10.0),\n  ('333445555', 20, 10.0),\n  ('999887777', 30, 30.0),\n  ('999887777', 10, 10.0),\n  ('987987987', 10, 35.0),\n  ('987987987', 30, 5.0),\n  ('987654321', 30, 20.0),\n  ('987654321', 20, 15.0),\n  ('888665555', 20, NULL);
""")
print("Đã tạo COMPANY từ Figure 5.6 của sách.")
print("DEPARTMENT:", conn.execute("SELECT COUNT(*) FROM DEPARTMENT").fetchone()[0], "dòng")
print("EMPLOYEE:", conn.execute("SELECT COUNT(*) FROM EMPLOYEE").fetchone()[0], "dòng")
print("PROJECT:", conn.execute("SELECT COUNT(*) FROM PROJECT").fetchone()[0], "dòng")
print("WORKS_ON:", conn.execute("SELECT COUNT(*) FROM WORKS_ON").fetchone()[0], "dòng")

Đã tạo COMPANY từ Figure 5.6 của sách.
DEPARTMENT: 3 dòng
EMPLOYEE: 8 dòng
PROJECT: 6 dòng
WORKS_ON: 16 dòng


In [ ]:
show("Bảng DEPARTMENT (Figure 5.6)", "SELECT * FROM DEPARTMENT ORDER BY Dnumber")
show("Nhân viên và phòng ban", "SELECT Fname, Lname, Salary, Dno FROM EMPLOYEE ORDER BY Ssn")


Bảng DEPARTMENT (Figure 5.6)
Dname          | Dnumber
---------------+--------
Headquarters   | 1      
Administration | 4      
Research       | 5      
Tổng: 3 dòng

Nhân viên và phòng ban
Fname    | Lname   | Salary | Dno
---------+---------+--------+----
John     | Smith   | 30000  | 5  
Franklin | Wong    | 40000  | 5  
Joyce    | English | 25000  | 5  
Ramesh   | Narayan | 38000  | 5  
James    | Borg    | 55000  | 1  
Jennifer | Wallace | 43000  | 4  
Ahmad    | Jabbar  | 25000  | 4  
Alicia   | Zelaya  | 25000  | 4  
Tổng: 8 dòng


[('John', 'Smith', 30000, 5),
 ('Franklin', 'Wong', 40000, 5),
 ('Joyce', 'English', 25000, 5),
 ('Ramesh', 'Narayan', 38000, 5),
 ('James', 'Borg', 55000, 1),
 ('Jennifer', 'Wallace', 43000, 4),
 ('Ahmad', 'Jabbar', 25000, 4),
 ('Alicia', 'Zelaya', 25000, 4)]

## 2. Khử lồng truy vấn với IN / Semi-join
**Theo tài liệu, mục 18.1.1 và 19.2.2:** `IN` có thể được chuyển về phép **semi-join**. Điều cốt lõi là kiểm tra *có tồn tại ít nhất một kết quả khớp*, không làm nhân bản dòng bên trái.

Ví dụ trong tài liệu dùng điều kiện `Salary > 200000`. Vì tất cả giá trị `Salary` ở **Figure 5.6** đều thấp hơn mức đó, ở phần chạy này dùng ngưỡng `Salary > 30000` (giá trị 30000 cũng xuất hiện ngay tại Figure 5.6) để nhìn thấy rõ kết quả và tình huống trùng dòng. Đây là thay đổi tham số minh họa, **không phải dữ liệu mới**.

- **Trước:** truy vấn `IN` chứa khối SELECT bên trong.
- **Sau:** biểu diễn hiệu ứng semi-join bằng `JOIN` với tập mã phòng ban **`DISTINCT`**. SQLite không có cú pháp `S=` trong sách, nên dùng SQL tương đương để minh họa.

In [ ]:
q_in = """
SELECT D.Dnumber, D.Dname
FROM DEPARTMENT D
WHERE D.Dnumber IN (
    SELECT E.Dno FROM EMPLOYEE E WHERE E.Salary > 30000
)
ORDER BY D.Dnumber
"""
q_semijoin_sql = """
SELECT D.Dnumber, D.Dname
FROM DEPARTMENT D
JOIN (SELECT DISTINCT Dno FROM EMPLOYEE WHERE Salary > 30000) E
  ON E.Dno = D.Dnumber
ORDER BY D.Dnumber
"""

before = show("TRƯỚC: IN (truy vấn lồng)", q_in)
after = show("SAU: biểu diễn semi-join bằng JOIN + DISTINCT", q_semijoin_sql)
assert before == after, "Hai truy vấn phải trả cùng dữ liệu!"
print("✓ Kết quả tương đương, không sinh dòng trùng.")


TRƯỚC: IN (truy vấn lồng)
Dnumber | Dname         
--------+---------------
1       | Headquarters  
4       | Administration
5       | Research      
Tổng: 3 dòng

SAU: biểu diễn semi-join bằng JOIN + DISTINCT
Dnumber | Dname         
--------+---------------
1       | Headquarters  
4       | Administration
5       | Research      
Tổng: 3 dòng
✓ Kết quả tương đương, không sinh dòng trùng.


**Vì sao không thay ngay bằng INNER JOIN thông thường?** Mục 19.2.2 trang sách 704, một phòng ban có thể khớp **nhiều** nhân viên. INNER JOIN có thể làm phòng ban xuất hiện nhiều lần và gây sai số khi `COUNT(*)`; semi-join không làm như vậy.

In [ ]:
q_join_naive = """
SELECT D.Dnumber, D.Dname
FROM DEPARTMENT D
JOIN EMPLOYEE E ON E.Dno = D.Dnumber
WHERE E.Salary > 30000
ORDER BY D.Dnumber
"""
naive = show("MINH HỌA LỖI: JOIN trực tiếp không khử trùng", q_join_naive)
print(f"Số dòng IN: {len(before)}; số dòng JOIN trực tiếp: {len(naive)}")
assert len(naive) > len(before)
print("✓ Điều này giải thích vì sao phải giữ ngữ nghĩa semi-join.")


MINH HỌA LỖI: JOIN trực tiếp không khử trùng
Dnumber | Dname         
--------+---------------
1       | Headquarters  
4       | Administration
5       | Research      
5       | Research      
Tổng: 4 dòng
Số dòng IN: 3; số dòng JOIN trực tiếp: 4
✓ Điều này giải thích vì sao phải giữ ngữ nghĩa semi-join.


## 3. Khử lồng một truy vấn con tương quan (EXISTS)
Theo mục 19.2.2, truy vấn `EXISTS` có tham chiếu từ truy vấn ngoài (`E.Dno`) có thể được đổi thành phép JOIN khi phù hợp. **Ở đây sử dụng phòng ban `Research` đã có trong Figure 5.6**, thay cho điều kiện `Zipcode = 30332` ở ví dụ gốc của mục 19.2.2, bởi bảng `DEPARTMENT` trong Figure 5.6 không có cột `Zipcode`.

Vì `Dnumber` là **khóa chính** nên mỗi nhân viên chỉ có thể ghép với tối đa một bản ghi phòng ban; phép INNER JOIN trong ví dụ này bảo toàn số dòng.

In [ ]:
q_exists = """
SELECT E.Fname, E.Lname, E.Salary
FROM EMPLOYEE E
WHERE EXISTS (
  SELECT 1 FROM DEPARTMENT D
  WHERE D.Dnumber = E.Dno AND D.Dname = 'Research'
)
ORDER BY E.Ssn
"""
q_unnested = """
SELECT E.Fname, E.Lname, E.Salary
FROM EMPLOYEE E
JOIN DEPARTMENT D ON E.Dno = D.Dnumber
WHERE D.Dname = 'Research'
ORDER BY E.Ssn
"""
a = show("TRƯỚC: correlated EXISTS", q_exists)
b = show("SAU: INNER JOIN", q_unnested)
assert a == b
print("✓ Hai truy vấn trả cùng nhân viên của phòng Research.")
show_plan("Execution plan mà SQLite thực tế chọn — EXISTS", q_exists)
show_plan("Execution plan mà SQLite thực tế chọn — INNER JOIN", q_unnested)


TRƯỚC: correlated EXISTS
Fname    | Lname   | Salary
---------+---------+-------
John     | Smith   | 30000 
Franklin | Wong    | 40000 
Joyce    | English | 25000 
Ramesh   | Narayan | 38000 
Tổng: 4 dòng

SAU: INNER JOIN
Fname    | Lname   | Salary
---------+---------+-------
John     | Smith   | 30000 
Franklin | Wong    | 40000 
Joyce    | English | 25000 
Ramesh   | Narayan | 38000 
Tổng: 4 dòng
✓ Hai truy vấn trả cùng nhân viên của phòng Research.

Execution plan mà SQLite thực tế chọn — EXISTS
   SCAN E USING INDEX sqlite_autoindex_EMPLOYEE_1
   CORRELATED SCALAR SUBQUERY 1
   SEARCH D USING INTEGER PRIMARY KEY (rowid=?)

Execution plan mà SQLite thực tế chọn — INNER JOIN
   SCAN E USING INDEX sqlite_autoindex_EMPLOYEE_1
   SEARCH D USING INTEGER PRIMARY KEY (rowid=?)


## 4. View Merging (theo Chương 19, mục 19.2.3)
Sách trình bày kỹ thuật hợp nhất một **inline view** (khối `SELECT` trong `FROM`) hoặc View thông thường vào câu truy vấn ngoài. Ví dụ của mục 19.2.3 có bảng `BLDG`, nhưng **Figure 5.6 không cung cấp dữ liệu cho BLDG**. Để không phải chế dữ liệu, phần chạy này dùng ba bảng **EMPLOYEE – WORKS_ON – PROJECT** và View `WORKS_ON1` xuất hiện thật trong **mục 7.3, trang sách 228–229**.

Yêu cầu: lấy tên nhân viên tham gia dự án `ProductX`. Đây cũng là giá trị dự án trong Figure 5.6.

In [ ]:
q_inline = """
SELECT E.Fname, E.Lname, V.Pname, V.Hours
FROM EMPLOYEE E
JOIN (
    SELECT W.Essn, P.Pname, W.Hours
    FROM WORKS_ON W
    JOIN PROJECT P ON W.Pno = P.Pnumber
) V ON E.Ssn = V.Essn
WHERE V.Pname = 'ProductX'
ORDER BY E.Ssn
"""
q_merged = """
SELECT E.Fname, E.Lname, P.Pname, W.Hours
FROM EMPLOYEE E
JOIN WORKS_ON W ON E.Ssn = W.Essn
JOIN PROJECT P ON W.Pno = P.Pnumber
WHERE P.Pname = 'ProductX'
ORDER BY E.Ssn
"""
v1 = show("TRƯỚC: Inline View trong FROM", q_inline)
v2 = show("SAU: gộp View vào khối truy vấn chính", q_merged)
assert v1 == v2
print("✓ Hai cách viết có kết quả giống hệt nhau.")
show_plan("Plan của SQLite — có inline view", q_inline)
show_plan("Plan của SQLite — view đã gộp", q_merged)


TRƯỚC: Inline View trong FROM
Fname | Lname   | Pname    | Hours
------+---------+----------+------
John  | Smith   | ProductX | 32.5 
Joyce | English | ProductX | 20.0 
Tổng: 2 dòng

SAU: gộp View vào khối truy vấn chính
Fname | Lname   | Pname    | Hours
------+---------+----------+------
John  | Smith   | ProductX | 32.5 
Joyce | English | ProductX | 20.0 
Tổng: 2 dòng
✓ Hai cách viết có kết quả giống hệt nhau.

Plan của SQLite — có inline view
   SCAN W
   SEARCH P USING INTEGER PRIMARY KEY (rowid=?)
   SEARCH E USING INDEX sqlite_autoindex_EMPLOYEE_1 (Ssn=?)
   USE TEMP B-TREE FOR ORDER BY

Plan của SQLite — view đã gộp
   SCAN W
   SEARCH P USING INTEGER PRIMARY KEY (rowid=?)
   SEARCH E USING INDEX sqlite_autoindex_EMPLOYEE_1 (Ssn=?)
   USE TEMP B-TREE FOR ORDER BY


### 4.1. Kiểm tra với View được định nghĩa bằng CREATE VIEW
`WORKS_ON1` dưới đây dựa trên **V1 của mục 7.3.2 (trang PDF 259)**. Trong SQL, việc View **được định nghĩa** không có nghĩa kết quả của nó đã được lưu sẵn: đó là nội dung cần phân biệt với Materialized View ở **mục 4**.

In [ ]:
conn.execute("""
CREATE VIEW WORKS_ON1 AS
SELECT E.Fname, E.Lname, P.Pname, W.Hours
FROM EMPLOYEE E
JOIN WORKS_ON W ON E.Ssn = W.Essn
JOIN PROJECT P ON P.Pnumber = W.Pno
""")
q_named_view = """
SELECT Fname, Lname, Pname, Hours
FROM WORKS_ON1
WHERE Pname = 'ProductX'
ORDER BY Fname, Lname, Hours
"""
q_direct = """
SELECT E.Fname, E.Lname, P.Pname, W.Hours
FROM EMPLOYEE E
JOIN WORKS_ON W ON E.Ssn = W.Essn
JOIN PROJECT P ON P.Pnumber = W.Pno
WHERE P.Pname = 'ProductX'
ORDER BY E.Fname, E.Lname, W.Hours
"""
c = show("Dùng WORKS_ON1", q_named_view)
d = show("Truy vấn trực tiếp bảng gốc", q_direct)
assert c == d
print("✓ View và truy vấn tương ứng trên bảng gốc trả cùng kết quả.")


Dùng WORKS_ON1
Fname | Lname   | Pname    | Hours
------+---------+----------+------
John  | Smith   | ProductX | 32.5 
Joyce | English | ProductX | 20.0 
Tổng: 2 dòng

Truy vấn trực tiếp bảng gốc
Fname | Lname   | Pname    | Hours
------+---------+----------+------
John  | Smith   | ProductX | 32.5 
Joyce | English | ProductX | 20.0 
Tổng: 2 dòng
✓ View và truy vấn tương ứng trên bảng gốc trả cùng kết quả.


## 5. Kết luận mục 3
- **Subquery Unnesting:** biến đổi cấu trúc truy vấn con để bộ tối ưu hóa có thêm phương án xử lý; phải giữ nguyên ngữ nghĩa truy vấn.
- **Semi-join / Anti-join:** phù hợp khi quan tâm *có / không có* dòng khớp (theo mục 18.1.1). Mô phỏng ở đây tập trung vào **semi-join**; không tự động đồng nhất mọi `NOT IN` với một `NOT EXISTS` nếu có `NULL`.
- **View Merging:** hợp nhất nội dung View vào khối truy vấn lớn để có thêm lựa chọn JOIN/chỉ mục; không bảo đảm mọi View phức tạp đều có thể gộp.
- **EXPLAIN QUERY PLAN:** là kết quả thực tế SQLite chọn khi chạy notebook. Hai cách viết có thể cho cùng một plan nếu DBMS đã tự tối ưu, **không nên kết luận chắc chắn cách viết mới luôn nhanh hơn**.

**Nguồn kiến thức:** Elmasri & Navathe, 7th ed., chương 7 mục 7.3, chương 18 mục 18.1.1, chương 19 mục 19.2.2–19.2.3.